In [16]:
# CSEC master module

import pandas as pd
from pathlib import Path


PROJECT_ROOT = Path("/")
RAW_ROOT = PROJECT_ROOT / "UgandaLSMS"

OUT_ROOT = PROJECT_ROOT / "Finished sections" / "Community"
OUT_ROOT.mkdir(parents=True, exist_ok=True)

WAVE_PATHS = {
    1: RAW_ROOT / "Wave_1",
    2: RAW_ROOT / "Wave_2",
    3: RAW_ROOT / "Wave_3",
    4: RAW_ROOT / "Wave_4",
    5: RAW_ROOT / "Wave_5",
    7: RAW_ROOT / "Wave_7",
    8: RAW_ROOT / "Wave_8",
}


def load_dta(path):
    df = pd.read_stata(path, convert_categoricals=False)
    df.columns = df.columns.str.strip().str.upper()
    return df


def find_file_case_insensitive(folder, filename):
    filename_upper = filename.upper()

    matches = [p for p in folder.iterdir() if p.name.upper() == filename_upper]

    if len(matches) == 0:
        raise FileNotFoundError(f"Could not find {filename} in {folder}")

    if len(matches) > 1:
        raise ValueError(f"Multiple matches for {filename} in {folder}: {matches}")

    return matches[0]


def build_source_variable_map(spec, source):
    source_col = source["source_col"]
    variable_map = {}

    for row in spec["crosswalk_rows"]:
        raw = row.get(source_col)

        if raw is None or str(raw).strip() == "":
            continue

        raw = str(raw).strip().upper()

        if row.get("is_key"):
            target = row["standard_name"].strip().upper()
        else:
            target = row["target"].strip().upper()

        variable_map[raw] = target

    return variable_map


def get_expected_cols(spec):
    expected_cols = []

    for row in spec["crosswalk_rows"]:
        if row.get("is_key"):
            expected_cols.append(row["standard_name"].strip().upper())
        else:
            expected_cols.append(row["target"].strip().upper())

    return list(dict.fromkeys(expected_cols))


def get_metadata_cols(spec):
    metadata_cols = ["WAVE", "SOURCE_FILE", "SOURCE_SECTION"]

    extra_metadata = spec.get("metadata_cols", [])

    for col in extra_metadata:
        col = col.upper()
        if col not in metadata_cols:
            metadata_cols.append(col)

    return metadata_cols

# ============================================================
# Section specific helper functions
#
# 1st: CSEC2A remove "no" observations
# 2nd: CSEC6C/NGO: community identifiers corrupted/constant. Fix by including temporary sequential community column counting/incrementing every time NGO_ID == 1
#
#
#
# ============================================================
# 1st:

def is_yes_or_available_value(x):
    if pd.isna(x):
        return False

    s = str(x).strip().upper()

    if s in ["", ".", "NAN", "NONE", "<NA>"]:
        return False

    return s in ["1", "1.0", "2", "2.0", "YES", "Y", "TRUE"]


def filter_csec2a_available_rows(df):
    if "C2AQ3" not in df.columns:
        return df

    df = df.copy()

    keep = df["C2AQ3"].map(is_yes_or_available_value)

    before = len(df)
    df = df.loc[keep].copy()
    after = len(df)

    print(f"CSEC2A availability filter: dropped {before - after:,} rows; kept {after:,}")

    return df


# ============================================================
#2nd:

def add_csec6c_generated_community_id(df_raw, source):
    df = df_raw.copy()

    if source["wave"] == 3 and "NGO_ID" in df.columns:
        ngo = pd.to_numeric(df["NGO_ID"], errors="coerce")
        df["GENERATED_COMMUNITY_ID"] = ngo.eq(1).cumsum()
        df.loc[ngo.isna(), "GENERATED_COMMUNITY_ID"] = pd.NA

    return df

def add_csec6c_wave2_ngo_id(df_raw, source):
    df = df_raw.copy()

    if source["wave"] == 2 and "COMCOD" in df.columns:
        df["NGO_ID"] = df.groupby("COMCOD", dropna=False).cumcount() + 1

    return df


def drop_empty_csec6c_ngo_rows(df, source):
    if source["wave"] not in [2, 4]:
        return df

    ngo_cols = [
        "NGO_NAME",
        "NGO_TYPE",
        "NGO_START_YEAR",
        "NGO_PURPOSE",
        "NGO_MEMBERS",
        "NGO_FEMALE_MEMBERS",
    ]

    existing_cols = [c for c in ngo_cols if c in df.columns]

    if not existing_cols:
        return df

    temp = df[existing_cols].copy()
    temp = temp.replace(["", " ", ".", "nan", "NaN", "None", "<NA>"], pd.NA)

    keep = temp.notna().any(axis=1)

    before = len(df)
    df = df.loc[keep].copy()
    after = len(df)

    print(f"CSEC6C Wave {source['wave']} empty NGO filter: dropped {before - after:,} rows; kept {after:,}")

    return df

# ============================================================



# ============================================================
# standardize var



def standardize_one_source(source, spec):
    wave = source["wave"]
    filename = source["file"]
    source_section = source["source_section"]
    variable_map = source["variable_map"]

    folder = WAVE_PATHS[wave]
    path = find_file_case_insensitive(folder, filename)

    df_raw = load_dta(path)

    if spec["section"] == "CSEC6C":
        df_raw = add_csec6c_wave2_ngo_id(df_raw, source)
        df_raw = add_csec6c_generated_community_id(df_raw, source)

    available_map = {
        raw: target
        for raw, target in variable_map.items()
        if raw in df_raw.columns
    }

    missing_raw = [
        raw
        for raw in variable_map
        if raw not in df_raw.columns
    ]

    df = df_raw[list(available_map.keys())].copy()
    df = df.rename(columns=available_map)

    duplicated_cols = df.columns[df.columns.duplicated()].tolist()
    if duplicated_cols:
        raise ValueError(
            f"Duplicate standardized columns in wave {wave}, "
            f"{source_section}: {duplicated_cols}"
        )

    expected_cols = get_expected_cols(spec)

    for col in expected_cols:
        if col not in df.columns:
            df[col] = pd.NA

    for key in spec["standard_keys"]:
        if key in df.columns:
            df[key] = df[key].astype("string").str.strip()

    df.insert(0, "SOURCE_SECTION", source_section)
    df.insert(0, "SOURCE_FILE", path.name)
    df.insert(0, "WAVE", wave)

    for meta_col in spec.get("metadata_cols", []):
        meta_col_upper = meta_col.upper()
        source_key = meta_col.lower()

        if source_key in source:
            df[meta_col_upper] = source[source_key]
        elif meta_col in source:
            df[meta_col_upper] = source[meta_col]
        else:
            df[meta_col_upper] = pd.NA

    metadata_cols = get_metadata_cols(spec)
    final_cols = metadata_cols + expected_cols
    df = df[final_cols]

    if spec["section"] == "CSEC6C":
        df = drop_empty_csec6c_ngo_rows(df, source)

    if spec["section"] == "CSEC2A":
        df = filter_csec2a_available_rows(df)

    print(f"\nLoaded Wave {wave}: {path.name}")
    print(f"Source section: {source_section}")
    print(f"Rows: {len(df):,}")
    print(f"Mapped variables found: {len(available_map):,}/{len(variable_map):,}")

    if missing_raw:
        print(f"Missing raw variables: {missing_raw}")

    return df


# ============================================================
# Stack

def stack_section(spec):
    section_name = spec["section"]
    pieces = []

    for source in spec["sources"]:
        try:
            df_source = standardize_one_source(source, spec)
            pieces.append(df_source)
        except FileNotFoundError as e:
            print(f"\nSKIPPED missing file: {e}")

    if not pieces:
        raise ValueError(f"No files loaded for {section_name}")

    df_stacked = pd.concat(pieces, ignore_index=True)

    print(f"\n==============================")
    print(f"STACKED {section_name}")
    print(f"Rows: {len(df_stacked):,}")
    print(f"Columns: {len(df_stacked.columns):,}")
    print("==============================")

    metadata_cols = get_metadata_cols(spec)

    group_cols = [
        c for c in metadata_cols
        if c in df_stacked.columns and c not in ["SOURCE_FILE", "SOURCE_SECTION"]
    ]

    if group_cols:
        print("\nRows by metadata:")
        print(
            df_stacked
            .groupby(group_cols, dropna=False)
            .size()
            .reset_index(name="ROWS")
            .to_string(index=False)
        )

    keys = spec["standard_keys"]
    dup_subset = group_cols + keys

    duplicate_count = df_stacked.duplicated(subset=dup_subset).sum()

    print(f"\nUnique metadata-key rows: {df_stacked[dup_subset].drop_duplicates().shape[0]:,}")
    print(f"Duplicates on metadata-keys: {duplicate_count:,}")

    return df_stacked

# ============================================================

def export_section(df, section_name):
    section_name = section_name.upper()

    parquet_path = OUT_ROOT / f"{section_name}_standardized.parquet"
    csv_path = OUT_ROOT / f"{section_name}_standardized.csv"
    excel_path = OUT_ROOT / f"{section_name}_standardized_preview.xlsx"

    df.to_csv(csv_path, index=False)

    preview_n = min(len(df), 10000)
    df.head(preview_n).to_excel(excel_path, index=False)

    df_parquet = df.copy()

    object_cols = df_parquet.select_dtypes(include=["object"]).columns

    for col in object_cols:
        df_parquet[col] = df_parquet[col].astype("string")

    df_parquet.to_parquet(parquet_path, index=False)

    print(f"\nExported:")
    print(f"Parquet: {parquet_path}")
    print(f"CSV:     {csv_path}")
    print(f"Excel preview first {preview_n:,} rows: {excel_path}")

    return parquet_path, csv_path, excel_path






In [8]:
# CSEC2A SPEC and out

CSEC2A_SPEC = {
    "section": "CSEC2A",
    "level": "community_service",
    "standard_keys": ["C2ASN"],

    "sources": [
        {"wave": 1, "file": "CSEC2A.dta", "source_section": "CSEC2A", "source_col": "W1"},
        {"wave": 2, "file": "CSECTION2A.dta", "source_section": "CSECTION2A", "source_col": "W2"},
        {"wave": 3, "file": "CSEC2A.dta", "source_section": "CSEC2A", "source_col": "W3"},
        {"wave": 4, "file": "CSEC2A.dta", "source_section": "CSEC2A", "source_col": "W4"},
        {"wave": 5, "file": "CSEC2A.dta", "source_section": "CSEC2A", "source_col": "W5"},
        {"wave": 7, "file": "CSEC2.dta", "source_section": "CSEC2", "source_col": "W7"},
        {"wave": 8, "file": "CSEC2.dta", "source_section": "CSEC2", "source_col": "W8"},
    ],

    "crosswalk_rows": [
        {
            "target": "DISTRICT",
            "is_key": False,
            "W1": None,
            "W2": None,
            "W3": "C1AQ1",
            "W4": None,
            "W5": None,
            "W7": None,
            "W8": None,
        },
        {
            "target": "COUNTY",
            "is_key": False,
            "W1": None,
            "W2": None,
            "W3": "C1AQ2",
            "W4": None,
            "W5": None,
            "W7": None,
            "W8": None,
        },
        {
            "target": "SUBCOUNTY",
            "is_key": False,
            "W1": None,
            "W2": None,
            "W3": "C1AQ3",
            "W4": None,
            "W5": None,
            "W7": None,
            "W8": None,
        },
        {
            "target": "PARISH",
            "is_key": False,
            "W1": None,
            "W2": None,
            "W3": "C1AQ4",
            "W4": None,
            "W5": None,
            "W7": None,
            "W8": None,
        },
        {
            "target": "VILLAGE_CODE",
            "is_key": False,
            "W1": None,
            "W2": None,
            "W3": None,
            "W4": "VILLAGECODE",
            "W5": "VILLAGECODE",
            "W7": None,
            "W8": None,
        },
        {
            "target": "INTERVIEW_KEY",
            "is_key": False,
            "W1": None,
            "W2": None,
            "W3": None,
            "W4": None,
            "W5": None,
            "W7": "INTERVIEW__KEY",
            "W8": "INTERVIEW__ID",
        },
        {
            "target": "EA_CODE",
            "is_key": False,
            "W1": None,
            "W2": None,
            "W3": None,
            "W4": None,
            "W5": None,
            "W7": "EA_CODE",
            "W8": "FINAL_EA_CODE",
        },
        {
            "target": "COMCOD",
            "is_key": False,
            "W1": "COMCOD",
            "W2": "COMCOD",
            "W3": None,
            "W4": None,
            "W5": None,
            "W7": None,
            "W8": None,
        },
        {
            "target": "C2ASN",
            "is_key": True,
            "standard_name": "C2ASN",
            "W1": "C2ASN",
            "W2": "C2ASN",
            "W3": "C2AQ2",
            "W4": "CFSERVICE_ID",
            "W5": "CFSERVICE_ID",
            "W7": "SERVICE_AVAILABILTY__ID",
            "W8": "SERVICE_AVAILABILTY__ID",
        },
        {
            "target": "C2AQ3",
            "is_key": False,
            "W1": "C2AQ3",
            "W2": "C2AQ3",
            "W3": "C2AQ3",
            "W4": "C2AQ3",
            "W5": "C2AQ3",
            "W7": "S2AQ03_1",
            "W8": "S2AQ03",
        },
        {
            "target": "C2AQ7",
            "is_key": False,
            "W1": "C2AQ7",
            "W2": "C2AQ7",
            "W3": "C2AQ7",
            "W4": "C2AQ7",
            "W5": "C2AQ7",
            "W7": "S2AQ07",
            "W8": None,
        },
        {
            "target": "C2AQ8",
            "is_key": False,
            "W1": "C2AQ8",
            "W2": "C2AQ8",
            "W3": "C2AQ8",
            "W4": "C2AQ8",
            "W5": "C2AQ8",
            "W7": "S2AQ08",
            "W8": None,
        },
    ],
}

for source in CSEC2A_SPEC["sources"]:
    source["variable_map"] = build_source_variable_map(CSEC2A_SPEC, source)

csec2a = stack_section(CSEC2A_SPEC)
export_section(csec2a, "CSEC2A")

csec2a.head()

CSEC2A availability filter: dropped 2,523 rows; kept 5,543

Loaded Wave 1: CSEC2A.dta
Source section: CSEC2A
Rows: 5,543
Mapped variables found: 5/5
CSEC2A availability filter: dropped 4,373 rows; kept 5,967

Loaded Wave 2: CSECTION2A.dta
Source section: CSECTION2A
Rows: 5,967
Mapped variables found: 5/5
CSEC2A availability filter: dropped 2,613 rows; kept 3,492

Loaded Wave 3: CSEC2a.dta
Source section: CSEC2A
Rows: 3,492
Mapped variables found: 8/8
CSEC2A availability filter: dropped 4,388 rows; kept 5,677

Loaded Wave 4: CSEC2A.dta
Source section: CSEC2A
Rows: 5,677
Mapped variables found: 5/5
CSEC2A availability filter: dropped 4,191 rows; kept 5,181

Loaded Wave 5: CSEC2A.dta
Source section: CSEC2A
Rows: 5,181
Mapped variables found: 5/5
CSEC2A availability filter: dropped 174 rows; kept 5,425

Loaded Wave 7: CSEC2.dta
Source section: CSEC2
Rows: 5,425
Mapped variables found: 6/6
CSEC2A availability filter: dropped 696 rows; kept 9,435

Loaded Wave 8: CSEC2.dta
Source section: CSE

C:\Users\Carl\AppData\Local\Temp\ipykernel_19384\2451754748.py:284: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  object_cols = df_parquet.select_dtypes(include=["object"]).columns


,WAVE,SOURCE_FILE,SOURCE_SECTION,DISTRICT,COUNTY,SUBCOUNTY,PARISH,VILLAGE_CODE,INTERVIEW_KEY,EA_CODE,COMCOD,C2ASN,C2AQ3,C2AQ7,C2AQ8
0,1,CSEC2A.dta,CSEC2A,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,1010002,1,2.0,120.0,2.0
1,1,CSEC2A.dta,CSEC2A,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,1010002,7,1.0,5.0,3.0
2,1,CSEC2A.dta,CSEC2A,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,1010002,10,1.0,10.0,1.0
3,1,CSEC2A.dta,CSEC2A,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,1010002,16,1.0,5.0,1.0
4,1,CSEC2A.dta,CSEC2A,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,1010002,24,1.0,0.0,3.0


In [10]:
# CSEC2C SPEC

CSEC2C_SPEC = {
    "section": "CSEC2C",
    "level": "community_service",
    "standard_keys": ["C2CQ15"],

    "sources": [
        {"wave": 1, "file": "CSEC2C.dta", "source_section": "CSEC2C", "source_col": "W1"},
        {"wave": 2, "file": "CSECTION2C.dta", "source_section": "CSECTION2C", "source_col": "W2"},
        {"wave": 3, "file": "CSEC2C_1.dta", "source_section": "CSEC2C_1", "source_col": "W3"},
        {"wave": 4, "file": "CSEC2C.dta", "source_section": "CSEC2C", "source_col": "W4"},
        {"wave": 5, "file": "CSEC2C_1.dta", "source_section": "CSEC2C_1", "source_col": "W5"},
        {"wave": 7, "file": "CSEC2C_0.dta", "source_section": "CSEC2C_0", "source_col": "W7"},
        {"wave": 8, "file": "CSEC2C_0.dta", "source_section": "CSEC2C_0", "source_col": "W8"},
    ],

    "crosswalk_rows": [
        {
            "target": "DISTRICT",
            "is_key": False,
            "W1": None,
            "W2": None,
            "W3": "C1AQ1",
            "W4": "DISTRICT_NAME",
            "W5": "DISTRICT_NAME",
            "W7": None,
            "W8": None,
        },
        {
            "target": "COUNTY_MUNICIPALITY",
            "is_key": False,
            "W1": None,
            "W2": None,
            "W3": "C1AQ2",
            "W4": None,
            "W5": None,
            "W7": None,
            "W8": None,
        },
        {
            "target": "SUBCOUNTY",
            "is_key": False,
            "W1": None,
            "W2": None,
            "W3": "C1AQ3",
            "W4": "SUBCOUNTY",
            "W5": "SUBCOUNTY",
            "W7": None,
            "W8": None,
        },
        {
            "target": "PARISH",
            "is_key": False,
            "W1": None,
            "W2": None,
            "W3": "C1AQ4",
            "W4": "PARISH",
            "W5": "PARISH",
            "W7": None,
            "W8": None,
        },
        {
            "target": "EA",
            "is_key": False,
            "W1": None,
            "W2": None,
            "W3": "C1AQ5",
            "W4": None,
            "W5": None,
            "W7": "EA_CODE",
            "W8": "FINAL_EA_CODE",
        },
        {
            "target": "VILLAGE_CODE",
            "is_key": False,
            "W1": None,
            "W2": None,
            "W3": None,
            "W4": "VILLAGECODE",
            "W5": "VILLAGECODE",
            "W7": "EA_CODE",
            "W8": None,
        },
        {
            "target": "LAST_WAVE_VILLAGE_CODE",
            "is_key": False,
            "W1": None,
            "W2": None,
            "W3": None,
            "W4": None,
            "W5": None,
            "W7": "T0_EA_CODE",
            "W8": None,
        },
        {
            "target": "INTERVIEW_KEY",
            "is_key": False,
            "W1": None,
            "W2": None,
            "W3": None,
            "W4": None,
            "W5": None,
            "W7": "INTERVIEW__KEY",
            "W8": "INTERVIEW__ID",
        },
        {
            "target": "COMCOD",
            "is_key": False,
            "W1": "COMCOD",
            "W2": "COMCOD",
            "W3": None,
            "W4": None,
            "W5": None,
            "W7": None,
            "W8": None,
        },
        {
            "target": "C2CQ15",
            "is_key": True,
            "standard_name": "C2CQ15",
            "W1": "C2CQ15",
            "W2": "C2CQ15",
            "W3": "C2CQ15",
            "W4": "C2CQ15",
            "W5": "C2CQ15",
            "W7": "S2CQ15",
            "W8": "S2CQ15",
        },
        {
            "target": "C2CQ18",
            "is_key": False,
            "W1": "C2CQ18",
            "W2": "C2CQ18",
            "W3": "C2CQ18",
            "W4": "C2CQ18",
            "W5": "C2CQ18",
            "W7": "S2CQ18",
            "W8": "S2CQ18",
        },
        {
            "target": "C2CQ19A",
            "is_key": False,
            "W1": "C2CQ19A",
            "W2": "C2CQ19A",
            "W3": "C2CQ19A",
            "W4": "C2CQ19A",
            "W5": "C2CQ19A",
            "W7": "S2CQ19A",
            "W8": "S2CQ19A",
        },
        {
            "target": "C2CQ19B",
            "is_key": False,
            "W1": "C2CQ19B",
            "W2": "C2CQ19B",
            "W3": "C2CQ19B",
            "W4": "C2CQ19B",
            "W5": "C2CQ19B",
            "W7": "S2CQ19B",
            "W8": "S2CQ19B",
        },
        {
            "target": "C2CQ19C",
            "is_key": False,
            "W1": "C2CQ19C",
            "W2": "C2CQ19C",
            "W3": "C2CQ19C",
            "W4": "C2CQ19C",
            "W5": "C2CQ19C",
            "W7": "S2CQ19C",
            "W8": "S2CQ19C",
        },
    ],
}

for source in CSEC2C_SPEC["sources"]:
    source["variable_map"] = build_source_variable_map(CSEC2C_SPEC, source)

csec2c = stack_section(CSEC2C_SPEC)
export_section(csec2c, "CSEC2C")

csec2c.head()


Loaded Wave 1: CSEC2C.dta
Source section: CSEC2C
Rows: 181
Mapped variables found: 6/6

Loaded Wave 2: CSECTION2C.dta
Source section: CSECTION2C
Rows: 322
Mapped variables found: 6/6

Loaded Wave 3: CSEC2c_1.dta
Source section: CSEC2C_1
Rows: 311
Mapped variables found: 10/10

Loaded Wave 4: CSEC2C.dta
Source section: CSEC2C
Rows: 306
Mapped variables found: 9/9

Loaded Wave 5: CSEC2C_1.dta
Source section: CSEC2C_1
Rows: 300
Mapped variables found: 9/9

Loaded Wave 7: CSEC2C_0.dta
Source section: CSEC2C_0
Rows: 305
Mapped variables found: 8/8

Loaded Wave 8: CSEC2C_0.dta
Source section: CSEC2C_0
Rows: 307
Mapped variables found: 7/7

STACKED CSEC2C
Rows: 2,032
Columns: 17

Rows by metadata:
 WAVE  ROWS
    1   181
    2   322
    3   311
    4   306
    5   300
    7   305
    8   307

Unique metadata-key rows: 34
Duplicates on metadata-keys: 1,998

Exported:
Parquet: C:\Users\Carl\Desktop\CSB_project\Finished sections\Community\CSEC2C_standardized.parquet
CSV:     C:\Users\Carl\Deskt

C:\Users\Carl\AppData\Local\Temp\ipykernel_19384\2451754748.py:284: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  object_cols = df_parquet.select_dtypes(include=["object"]).columns


,WAVE,SOURCE_FILE,SOURCE_SECTION,DISTRICT,COUNTY_MUNICIPALITY,SUBCOUNTY,PARISH,EA,VILLAGE_CODE,LAST_WAVE_VILLAGE_CODE,INTERVIEW_KEY,COMCOD,C2CQ15,C2CQ18,C2CQ19A,C2CQ19B,C2CQ19C
0,1,CSEC2C.dta,CSEC2C,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,1020001,1,1.0,100.0,1.0,2.0
1,1,CSEC2C.dta,CSEC2C,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,1020006,1,1.0,100.0,1.0,2.0
2,1,CSEC2C.dta,CSEC2C,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,1020007,1,1.0,100.0,1.0,2.0
3,1,CSEC2C.dta,CSEC2C,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,1020008,1,1.0,100.0,1.0,2.0
4,1,CSEC2C.dta,CSEC2C,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,1020011,2,2.0,NaN,NaN,NaN


In [17]:
# CSEC6C/NGO module SPEC

CSEC6C_SPEC = {
    "section": "CSEC6C",
    "level": "community_ngo",
    "standard_keys": ["NGO_ID"],

    "sources": [
        {"wave": 2, "file": "CSECTION6C.dta", "source_section": "CSECTION6C", "source_col": "W2"},
        {"wave": 3, "file": "CSEC6C.dta", "source_section": "CSEC6C", "source_col": "W3"},
        {"wave": 4, "file": "CSEC6C.dta", "source_section": "CSEC6C", "source_col": "W4"},
        {"wave": 5, "file": "CSEC6C.dta", "source_section": "CSEC6C", "source_col": "W5"},
        {"wave": 7, "file": "CSEC6C_1.dta", "source_section": "CSEC6C_1", "source_col": "W7"},
        {"wave": 8, "file": "CSEC6C_1.dta", "source_section": "CSEC6C_1", "source_col": "W8"},
    ],

    "crosswalk_rows": [
        {"target": "DISTRICT", "is_key": False, "W2": None, "W3": "C1AQ1", "W4": "DISTRICT_NAME", "W5": "DISTRICT_NAME", "W7": None, "W8": None},
        {"target": "COUNTY", "is_key": False, "W2": None, "W3": "C1AQ2", "W4": None, "W5": None, "W7": None, "W8": None},
        {"target": "SUBCOUNTY", "is_key": False, "W2": None, "W3": "C1AQ3", "W4": "SUBCOUNTY", "W5": "SUBCOUNTY", "W7": None, "W8": None},
        {"target": "PARISH", "is_key": False, "W2": None, "W3": "C1AQ4", "W4": "PARISH", "W5": "PARISH", "W7": None, "W8": None},
        {"target": "VILLAGE_CODE", "is_key": False, "W2": None, "W3": None, "W4": "VILLAGECODE", "W5": "VILLAGECODE", "W7": None, "W8": None},
        {"target": "EA_CODE", "is_key": False, "W2": None, "W3": None, "W4": None, "W5": None, "W7": "EA_CODE", "W8": "FINAL_EA_CODE"},
        {"target": "INTERVIEW_KEY", "is_key": False, "W2": None, "W3": None, "W4": None, "W5": None, "W7": "INTERVIEW__KEY", "W8": "INTERVIEW__ID"},
        {"target": "COMCOD", "is_key": False, "W2": "COMCOD", "W3": None, "W4": None, "W5": None, "W7": None, "W8": None},
        {"target": "GENERATED_COMMUNITY_ID", "is_key": False, "W2": None, "W3": "GENERATED_COMMUNITY_ID", "W4": None, "W5": None, "W7": None, "W8": None},

        {"target": "NGO_ID", "is_key": True, "standard_name": "NGO_ID", "W2": "NGO_ID", "W3": "NGO_ID", "W4": "NGO_ID", "W5": "NGO_ID", "W7": "NGO_ID", "W8": "NGO_ID"},
        {"target": "NGO_NAME", "is_key": False, "W2": "C6CNGO", "W3": "NGO_NAME", "W4": None, "W5": None, "W7": "S6CQ0", "W8": None},
        {"target": "NGO_TYPE", "is_key": False, "W2": "C6CQ1", "W3": "C6CQ1", "W4": "C6CQ1", "W5": "C6CQ1", "W7": "S6CQ01", "W8": "S6CQ01"},
        {"target": "NGO_START_YEAR", "is_key": False, "W2": "C6CQ2", "W3": "C6CQ2", "W4": "C6CQ2", "W5": "C6CQ2", "W7": "S6CQ02", "W8": "S6CQ02"},
        {"target": "NGO_PURPOSE", "is_key": False, "W2": "C6CQ3", "W3": "C6CQ3", "W4": "C6CQ3", "W5": "C6CQ3", "W7": "S6CQ03", "W8": "S6CQ03"},
        {"target": "NGO_MEMBERS", "is_key": False, "W2": "C6CQ4", "W3": "C6CQ4", "W4": "C6CQ4", "W5": "C6CQ4", "W7": "S6CQ04", "W8": "S6CQ04"},
        {"target": "NGO_FEMALE_MEMBERS", "is_key": False, "W2": "C6CQ5", "W3": "C6CQ5", "W4": "C6CQ5", "W5": "C6CQ5", "W7": "S6CQ05", "W8": "S6CQ05"},
    ],
}




In [18]:
# CSEC6C/NGO out

for source in CSEC6C_SPEC["sources"]:
    source["variable_map"] = build_source_variable_map(CSEC6C_SPEC, source)

csec6c = stack_section(CSEC6C_SPEC)
export_section(csec6c, "CSEC6C")

csec6c.head()



CSEC6C Wave 2 empty NGO filter: dropped 286 rows; kept 177

Loaded Wave 2: CSECTION6C.dta
Source section: CSECTION6C
Rows: 177
Mapped variables found: 8/8

Loaded Wave 3: CSEC6c.dta
Source section: CSEC6C
Rows: 111
Mapped variables found: 12/12
CSEC6C Wave 4 empty NGO filter: dropped 197 rows; kept 173

Loaded Wave 4: CSEC6C.dta
Source section: CSEC6C
Rows: 173
Mapped variables found: 10/10

Loaded Wave 5: CSEC6C.dta
Source section: CSEC6C
Rows: 112
Mapped variables found: 10/10

Loaded Wave 7: CSEC6C_1.dta
Source section: CSEC6C_1
Rows: 71
Mapped variables found: 9/9

Loaded Wave 8: CSEC6C_1.dta
Source section: CSEC6C_1
Rows: 90
Mapped variables found: 8/8

STACKED CSEC6C
Rows: 734
Columns: 19

Rows by metadata:
 WAVE  ROWS
    2   177
    3   111
    4   173
    5   112
    7    71
    8    90

Unique metadata-key rows: 41
Duplicates on metadata-keys: 693

Exported:
Parquet: C:\Users\Carl\Desktop\CSB_project\Finished sections\Community\CSEC6C_standardized.parquet
CSV:     C:\Users\Ca

C:\Users\Carl\AppData\Local\Temp\ipykernel_19384\624961513.py:344: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  object_cols = df_parquet.select_dtypes(include=["object"]).columns


,WAVE,SOURCE_FILE,SOURCE_SECTION,DISTRICT,COUNTY,SUBCOUNTY,PARISH,VILLAGE_CODE,EA_CODE,INTERVIEW_KEY,COMCOD,GENERATED_COMMUNITY_ID,NGO_ID,NGO_NAME,NGO_TYPE,NGO_START_YEAR,NGO_PURPOSE,NGO_MEMBERS,NGO_FEMALE_MEMBERS
0,2,CSECTION6C.dta,CSECTION6C,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,1020004,<NA>,1,WATER AID,3.0,2009.0,1.0,0.0,0.0
1,2,CSECTION6C.dta,CSECTION6C,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,1020005,<NA>,1,CHAIN,3.0,2006.0,4.0,10.0,10.0
2,2,CSECTION6C.dta,CSECTION6C,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,1020006,<NA>,1,CIDI,2.0,2010.0,96.0,2.0,0.0
3,2,CSECTION6C.dta,CSECTION6C,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,1020007,<NA>,1,HADEC,1.0,2008.0,1.0,10.0,3.0
4,2,CSECTION6C.dta,CSECTION6C,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,1020008,<NA>,1,BAMBEJJA,3.0,2006.0,4.0,50.0,50.0
